<a href="https://colab.research.google.com/github/9terry-student/TensorFlow/blob/main/mamba_pytorch_minimal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MambaBlock(nn.Module):
  def __init__(self,d_model,d_state):
    super().__init__()
    self.d_model=d_model
    self.d_state=d_state

    # A: 상태 전이 행렬 (d_state,)
    self.A=nn.Parameter(torch.arange(1,d_state+1).float())

    # B,C: 입출력 행렬
    self.B=nn.Linear(d_model,d_state,bias=False)
    self.C=nn.Linear(d_state,d_model,bias=False)

    # delta: 이산화 파라미터
    self.delta=nn.Linear(d_model,d_state,bias=False)

    # gate: 정규화
    self.gate=nn.Linear(d_model,d_model)

  def forward(self,x):
    # x:(batch,seq_len,d_model)
    batch,seq_len,d_model=x.shape

    # delta 계산
    delta=F.softplus(self.delta(x))

    # A 이산화
    A=-torch.exp(self.A)      # 안정성을 위해 음수
    A_bar=torch.exp(delta*A)

    # B 이산화
    B=self.B(x)
    B_bar=delta*B

    # SSM 계산
    h=torch.zeros(batch,self.d_state)
    outputs=[]

    for t in range(seq_len):
      h=A_bar[:,t,:]*h+B_bar[:,t,:]*x[:,t,:1]
      y=self.C(h)
      outputs.append(y)

    output=torch.stack(outputs,dim=1)

    gate=F.silu(self.gate(x))
    output*=gate
    return output

In [2]:
# test
d_model=16
d_state=8
seq_len=4
batch=1

model=MambaBlock(d_model,d_state)
x=torch.randn(batch,seq_len,d_model)
output=model(x)
print('output shape: ',output.shape)      # (1,4,16)

output shape:  torch.Size([1, 4, 16])
